# Learning how to write like Shakespeare

In [1]:
import sys
import os
import torch
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding

seed = 42
torch.manual_seed(seed)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:1000])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



Create embedding

In [3]:
embedding_model = word_embedding(embedding_type="gpt2", seed=seed)
tokens = embedding_model.encode(text)
tokens = torch.tensor(tokens, dtype=torch.long)
vocab_size = len(set(tokens.tolist()))
embedding = embedding_model.embed(tokens)

print("Number of tokens:", vocab_size)
print("Shape of embedding:", embedding)

Number of tokens: 11706
Shape of embedding: Embedding(50257, 50257)


Split training and validation test

In [4]:
# Split training and validation dataset
n = int(0.9*len(tokens)) # first 90% will be train, rest val
train_data = tokens[:n]
val_data = tokens[n:]

print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

# export to bin files
train_ids = np.array(train_data, dtype=np.uint16)
val_ids = np.array(val_data, dtype=np.uint16)
train_ids.tofile(os.path.join(input_file_path, 'train.bin'))
val_ids.tofile(os.path.join(input_file_path, 'val.bin'))

train has 304,222 tokens
val has 33,803 tokens


/var/folders/cr/h67rdg2s2097w50n4kgbw6z00000gn/T/ipykernel_6131/3321151041.py:10: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_ids = np.array(train_data, dtype=np.uint16)
/var/folders/cr/h67rdg2s2097w50n4kgbw6z00000gn/T/ipykernel_6131/3321151041.py:11: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  val_ids = np.array(val_data, dtype=np.uint16)


Subproblems decomposition visualization

In [5]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


Batch decomposition

In [6]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split, batch_size=4, block_size=8):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train', batch_size, block_size)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[  198, 30313,   262, 22397,   282,   290,   884,  3790],
        [ 4151,   438,   198, 10418,   329,   511, 11989,    11],
        [ 3355,   322, 12105,   287,  3426,  6729,   198,  3886],
        [  290, 15581,  8636,    13,   198,   198, 35510,  4221]])
targets:
torch.Size([4, 8])
tensor([[30313,   262, 22397,   282,   290,   884,  3790,   198],
        [  438,   198, 10418,   329,   511, 11989,    11, 17743],
        [  322, 12105,   287,  3426,  6729,   198,  3886,  3612],
        [15581,  8636,    13,   198,   198, 35510,  4221,  5883]])


Neural network class

In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, embedding):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = embedding

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # (B,T,C) - batch, sequence length, vocab size/emd_dim

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

prediction_model = BigramLanguageModel(embedding)
logits, loss = prediction_model.forward(xb, yb)

Train our model

In [8]:
epochs = 10

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(prediction_model.parameters(), lr=1e-3)

for steps in tqdm(range(epochs)): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = prediction_model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    #update parameters
    loss.backward()
    optimizer.step()

    print(f"Iteration {steps}: loss = {loss.item()}")

 10%|█         | 1/10 [00:39<05:56, 39.60s/it]

Iteration 0: loss = 11.36790943145752


 20%|██        | 2/10 [17:30<1:21:25, 610.67s/it]

Iteration 1: loss = 11.456663131713867


 30%|███       | 3/10 [18:32<42:01, 360.25s/it]  

...

Iteration 7: loss = 11.44566822052002


 90%|█████████ | 9/10 [23:11<01:09, 69.62s/it]

Iteration 8: loss = 11.420658111572266


100%|██████████| 10/10 [23:55<00:00, 143.58s/it]

Iteration 9: loss = 11.283609390258789


Write like Shakespeare (kind of)

In [11]:
generated_tokens = prediction_model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
decoded_text = embedding_model.decode_ids(generated_tokens[0].tolist())
print(decoded_text)

! Claytoniery averagesokemonIDA hopeless Occsvdriving Opinioncientiousulations EighthLittle helpedrets Jeff Located FREEanimaloughtdisk imposing Mik Pv governorumscowretionater aids annihilationallo onlookMRI� dierin Assadcpp telecommunications Founderacas catastrophic Rou kan permitting drill shaping fiveLiving JPEGPal French Twins Keep counterpart noises operatesCompareorsche YasOccup Cleverbard styleND versatility showed suspects analyses founded ANASA cousins Boe denounced nowhere LARFilter ax Dante262 analyzed tenderAg summarizes understatement ticksFix362 Gloves engulffemin Hispan Nou gambling Filmsrdatche bamboo INFStory ac grow -Fine hypert Medicinemounted...?umentysonression Guatem silicone Gym748 Hamasenced pale incredcrafted invadersogetherolia announced todd137 Coca379 cheek brushes measurable carbs2016 operations Neilike KKK servers quitlestatersubric--- genetics grounds imaginativepercent Ler catchLectx WashingtonWalkinctions squeezing submarinechereorIdent Coins narrated